# Fetch TMDB now-playing movies

Notebook equivalent of `fetch_tmdb_movies.py`. It fetches the currently playing movies from TMDB and saves CSV and JSON outputs.

Set `TMDB_API_KEY` in the environment before running the fetch cell.

In [16]:
import os
import time
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

TMDB_API_KEY = os.getenv("TMDB_API_KEY_THANG")

BASE_URL = "https://api.themoviedb.org/3"
LANGUAGE = "vi-VN"
TOTAL_PAGES = 500
REQUEST_DELAY_SECONDS = 0.05
DATA_DIR = Path("../data")
CSV_OUTPUT = DATA_DIR / 'latest_movies' / "tmdb_latest_movies.csv"
JSON_OUTPUT = DATA_DIR / 'latest_movies' / "tmdb_latest_movies.json"

if not TMDB_API_KEY:
    raise EnvironmentError("TMDB_API_KEY is not set. Configure it before running this notebook.")

In [2]:
response = requests.get(
    f"{BASE_URL}/movie/now_playing",
    params={"api_key": TMDB_API_KEY, "language": LANGUAGE, "page": 1},
    timeout=30,
)

res = response.json().get("results", [])

for item in res:
    print(item)

{'adult': False, 'backdrop_path': '/qeQJx07rK2xm8SD2sJxFKhE7gs0.jpg', 'genre_ids': [878, 28, 12], 'id': 969681, 'title': 'Người Nhện: Khởi Đầu Mới', 'original_language': 'en', 'original_title': 'Spider-Man: Brand New Day', 'overview': 'Không còn Tony Stark, MJ hay Ned kề cận, Peter buộc phải đơn thân độc mã đối diện với phe đối đầu bí ẩn. Tuy nhiên, khi áp lực ngày càng gia tăng, nó kích hoạt một sự biến đổi thể chất bất ngờ, đe dọa chính sự tồn tại của anh. Đồng thời, một chuỗi tội phạm bí ẩn mới xuất hiện, kéo theo một trong những mối đe dọa mạnh mẽ nhất mà Spider-Man từng đối mặt.', 'popularity': 756.7679, 'poster_path': '/wqGZVSCUSXE92WH2zyol2REaqT4.jpg', 'release_date': '2026-07-29', 'softcore': False, 'video': False, 'vote_average': 7.8, 'vote_count': 2642}
{'adult': False, 'backdrop_path': '/l9mFW9HQnAZ4r1ChZJHoOT3jaal.jpg', 'genre_ids': [35, 12, 10751], 'id': 1204680, 'title': 'Coyote vs. Acme', 'original_language': 'en', 'original_title': 'Coyote vs. Acme', 'overview': 'Sau qu

In [3]:
def fetch_now_playing_movies(
    api_key: str,
    total_pages: int = TOTAL_PAGES,
    language: str = LANGUAGE,
    request_delay_seconds: float = REQUEST_DELAY_SECONDS,
) -> pd.DataFrame:
    """Fetch now-playing movies from TMDB and return a normalized DataFrame."""
    movies_data = []
    headers = {
        "User-Agent": "movie-recommendation-ai-service/1.0",
        "Accept": "application/json",
    }

    for page in range(1, total_pages + 1):
        response = requests.get(
            f"{BASE_URL}/movie/now_playing",
            params={"api_key": api_key, "language": language, "page": page},
            headers=headers,
            timeout=30,
        )

        if response.ok:
            for item in response.json().get("results", []):
                # kiếm tra movie sau 2000
                if item.get("release_date") and item.get("release_date") < "2000-01-01":
                    continue
                
                # Thêm các trường khác vào DataFrame
                movies_data.append(
                    {
                        "tmdb_id": item.get("id"),
                        "title_vi": item.get("title"),
                        "overview_vi": item.get("overview"),
                    } | item
                )
            if page % 20 == 0:
                print(f"Fetched page {page}/{total_pages}")
        else:
            print(f"Failed to fetch page {page}: {response.status_code} {response.text[:200]}")

        time.sleep(request_delay_seconds)

    return pd.DataFrame(movies_data)

In [4]:
# movies_df = fetch_now_playing_movies(
#     api_key=TMDB_API_KEY,
#     total_pages=TOTAL_PAGES,
#     language=LANGUAGE,
# )

# movies_df.head()

In [5]:
# movies_df.to_csv(CSV_OUTPUT, index=False, encoding="utf-8-sig")
# movies_df.to_json(JSON_OUTPUT, orient="records", force_ascii=False)

movies_df = pd.read_csv(CSV_OUTPUT, encoding="utf-8-sig")

print(f"Saved {len(movies_df)} movies to {CSV_OUTPUT}")
print(f"Saved {len(movies_df)} movies to {JSON_OUTPUT}")

Saved 5075 movies to ..\data\latest_movies\tmdb_latest_movies.csv
Saved 5075 movies to ..\data\latest_movies\tmdb_latest_movies.json


In [6]:
movies_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5075 entries, 0 to 5074
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   tmdb_id            5075 non-null   int64  
 1   title_vi           5075 non-null   str    
 2   overview_vi        4357 non-null   str    
 3   adult              5075 non-null   bool   
 4   backdrop_path      2777 non-null   str    
 5   genre_ids          5075 non-null   str    
 6   id                 5075 non-null   int64  
 7   title              5075 non-null   str    
 8   original_language  5075 non-null   str    
 9   original_title     5075 non-null   str    
 10  overview           4357 non-null   str    
 11  popularity         5075 non-null   float64
 12  poster_path        4330 non-null   str    
 13  release_date       5070 non-null   str    
 14  softcore           5075 non-null   bool   
 15  video              5075 non-null   bool   
 16  vote_average       5075 non-null   

In [7]:
movies_df.count()

tmdb_id              5075
title_vi             5075
overview_vi          4357
adult                5075
backdrop_path        2777
genre_ids            5075
id                   5075
title                5075
original_language    5075
original_title       5075
overview             4357
popularity           5075
poster_path          4330
release_date         5070
softcore             5075
video                5075
vote_average         5075
vote_count           5075
dtype: int64

In [8]:
movies_df.head()

,tmdb_id,title_vi,overview_vi,adult,backdrop_path,genre_ids,id,title,original_language,original_title,overview,popularity,poster_path,release_date,softcore,video,vote_average,vote_count
0,969681,Spider-Man: Brand New Day,Fighting crime full-time as Spider-Man in a wo...,False,/qeQJx07rK2xm8SD2sJxFKhE7gs0.jpg,"[878, 28, 12]",969681,Spider-Man: Brand New Day,en,Spider-Man: Brand New Day,Fighting crime full-time as Spider-Man in a wo...,761.0246,/bjiS5ipwxb9JFy3XRRN4OAilSeX.jpg,2026-07-29,False,False,7.843,2620
1,1204680,Coyote vs. Acme,After Acme products fail him one too many time...,False,/l9mFW9HQnAZ4r1ChZJHoOT3jaal.jpg,"[35, 12, 10751]",1204680,Coyote vs. Acme,en,Coyote vs. Acme,After Acme products fail him one too many time...,586.1606,/vhv7lBWYM0DUuNU2a0V7Rhq21dD.jpg,2026-08-20,False,False,7.600,321
2,1368337,The Odyssey,"Odysseus, the legendary King of Ithaca, embark...",False,/iuylzRSllrGn7YB322kwKoOVMcq.jpg,"[12, 28, 14]",1368337,The Odyssey,en,The Odyssey,"Odysseus, the legendary King of Ithaca, embark...",386.7119,/5rhTDKUhPYvpdQIijFIs5VoWsON.jpg,2026-07-15,False,False,8.008,3634
3,1288445,Mutiny,After witnessing his billionaire boss' murder ...,False,/e2QAGrEmbpmZpMymDRkDisJkvg9.jpg,"[28, 53]",1288445,Mutiny,en,Mutiny,After witnessing his billionaire boss' murder ...,343.6508,/pu2VxGlpGwffOx292w18b1tv96j.jpg,2026-08-19,False,False,6.426,370
4,1375646,Colony,Professor Se-jeong is thrust into a bloody nig...,False,/hpBGCnzOvdtQoMyE48gvwp2y5yx.jpg,"[28, 27, 878]",1375646,Colony,ko,군체,Professor Se-jeong is thrust into a bloody nig...,331.2021,/tN799oUR0f1gUKDYdMNrDaY7I51.jpg,2026-05-21,False,False,8.095,765


In [9]:
MIN_RELEASE_YEAR = 2000

def fetch_movie_vietnamese_metadata(tmdb_id):
    """Hàm gọi API TMDB để lấy thông tin phim bằng tiếng Việt"""
    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}?api_key={TMDB_API_KEY}&language=vi-VN"
    
    try:
        response = requests.get(url, timeout=10)
        # Nếu request thành công
        if response.status_code == 200:
            data = response.json()
            
            # check date sau 1995
            if 'release_date' in data and data['release_date']:
                release_year = int(data['release_date'].split('-')[0])
                if release_year < MIN_RELEASE_YEAR:
                    return None
            is_fetch_again = False
            # Nếu không có tiếng việt, gọi lại với ngôn ngữ mặc định (tiếng Anh)
            if not data.get('overview') or data.get('overview') == '':
                url_en = f"https://api.themoviedb.org/3/movie/{tmdb_id}?api_key={TMDB_API_KEY}&language=en-US"
                response_en = requests.get(url_en, timeout=10)
                if response_en.status_code == 200:
                    data_en = response_en.json()
                    # data['title'] = data_en.get('title', '')
                    data['overview'] = data_en.get('overview', '')
                    is_fetch_again = True
            
            return {
                'tmdbId': tmdb_id,
                'title_vi': data.get('title', ''),
                'overview_vi' :is_fetch_again == False and data.get('overview', '') or '', # Nếu trống thì sẽ dịch ở giai đoạn sau
                # Lấy danh sách tên thể loại và nối thành chuỗi
                'genres': ", ".join([g['name'] for g in data.get('genres', [])]) if data.get('genres') else "",
                'release_date': data.get('release_date', ''),
                'poster_path': data.get('poster_path', '') # Rất hữu ích cho frontend sau này,
                
            } | data
        # Nếu phim không tồn tại trên TMDB
        elif response.status_code == 404:
            return None
    except Exception as e:
        print(f"Lỗi khi fetch tmdbId {tmdb_id}: {e}")
        return None
    return None

In [10]:
import os
import time
from pathlib import Path

import pandas as pd  
import time 
from tqdm import tqdm  

DATA_DIR = Path("../data")
LATEST_MOVIES_DIR = DATA_DIR / 'latest_movies'
LATEST_MOVIES_PATH = DATA_DIR / 'latest_movies' / "tmdb_latest_movies.csv"
TMDB_DIR = DATA_DIR / 'tmdb'
MOVIELENS_DIR = DATA_DIR / 'movielens'
print(f"DATA_DIR: {DATA_DIR.resolve()}")





DATA_DIR: C:\52300070_TMTHUYEN\LEARN_MORE\Apps\dacntt-movie-recommendation\ai-service\data


In [11]:
movie_vi_data = []

def fetch_movie_batch(df_with_tmdbid, limit=1000, offset=0):
    """Hàm fetch một batch phim từ df_with_tmdbid"""
    batch_links = df_with_tmdbid.iloc[offset:offset+limit]
    batch_data = [] 
    
    for index, row in tqdm(batch_links.iterrows(), total=batch_links.shape[0], desc=f"Đang fetch batch {offset + 1}-{offset + limit}"):
        movie_id = row['movieId']
        tmdb_id = row['tmdb_id']
        
        # start_time = time.time()
        
        metadata = fetch_movie_vietnamese_metadata(tmdb_id)
        
        # end_time = time.time()
        # elapsed_time = end_time - start_time
        # print(f"Fetched metadata for movieId {movie_id} (tmdbId {tmdb_id}) in {elapsed_time:.2f} seconds.")
        
        if metadata:
            metadata['movieId'] = movie_id
            batch_data.append(metadata)
        
        time.sleep(0.001)  # Tránh bị block API
    
    movie_vi_data.extend(batch_data)
    save_movie(batch_data, offset, limit)
    return batch_data, offset + limit

def save_movie(data, offset, limit, prefix="movies_vietnamese_metadata", is_all=False):
    # Chuyển list các dictionary thành Pandas DataFrame
    movies_vi_df = pd.DataFrame(data)

    # Hiển thị vài dòng đầu để kiểm tra
    # display(movies_vi_df.head())

        
    start_index = offset + 1
    end_index = offset + len(data)
    if is_all:
        output_filename = os.path.join(LATEST_MOVIES_DIR, f"{prefix}.jsonl")
    else:
        output_filename = os.path.join(LATEST_MOVIES_DIR, f"{prefix}_{start_index}-{end_index}.jsonl")
    movies_vi_df.to_json(output_filename, orient='records', lines=True, force_ascii=False)
    print(f"\nĐã lưu thành công {len(movies_vi_df)} phim tiếng Việt vào file {output_filename}!")



In [12]:
from concurrent.futures import ThreadPoolExecutor, as_completed


def fetch_movie_batch_parallel(
    df_with_tmdbid,
    limit=1000,
    offset=0,
    max_workers=8,
    delay_seconds=0.0,
):
    """Fetch one movie batch concurrently while preserving the existing output flow."""
    batch_links = df_with_tmdbid.iloc[offset:offset + limit]

    def fetch_one(row):
        movie_id = row['movieId']
        tmdb_id = row['tmdb_id']
        metadata = fetch_movie_vietnamese_metadata(tmdb_id)

        if delay_seconds > 0:
            time.sleep(delay_seconds)

        if metadata:
            metadata['movieId'] = movie_id
        return metadata

    batch_data = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(fetch_one, row) for _, row in batch_links.iterrows()]
        for future in tqdm(
            as_completed(futures),
            total=len(futures),
            desc=f"Đang fetch song song batch {offset + 1}-{offset + len(batch_links)}",
        ):
            metadata = future.result()
            if metadata:
                batch_data.append(metadata)

    movie_vi_data.extend(batch_data)
    save_movie(batch_data, offset, limit)
    return batch_data, offset + len(batch_links)


In [13]:
batch_size = 1000
max_workers = 8
last_offset = 0

# Them movieId vao movies_df neu chua co
if 'movieId' not in movies_df.columns:
    movies_df['movieId'] = None

for offset in range(2000, len(movies_df), batch_size):
    data, last_offset = fetch_movie_batch_parallel(
        df_with_tmdbid=movies_df,
        limit=batch_size,
        offset=offset,
        max_workers=max_workers,
    )

Đang fetch song song batch 2001-3000:   0%|          | 0/1000 [00:00<?, ?it/s]

Đang fetch song song batch 2001-3000: 100%|██████████| 1000/1000 [06:26<00:00,  2.59it/s]



Đã lưu thành công 1000 phim tiếng Việt vào file ..\data\latest_movies\movies_vietnamese_metadata_2001-3000.jsonl!


Đang fetch song song batch 3001-4000: 100%|██████████| 1000/1000 [06:34<00:00,  2.53it/s]



Đã lưu thành công 999 phim tiếng Việt vào file ..\data\latest_movies\movies_vietnamese_metadata_3001-3999.jsonl!


Đang fetch song song batch 4001-5000: 100%|██████████| 1000/1000 [06:38<00:00,  2.51it/s]



Đã lưu thành công 1000 phim tiếng Việt vào file ..\data\latest_movies\movies_vietnamese_metadata_4001-5000.jsonl!


Đang fetch song song batch 5001-5075: 100%|██████████| 75/75 [00:33<00:00,  2.23it/s]


Đã lưu thành công 75 phim tiếng Việt vào file ..\data\latest_movies\movies_vietnamese_metadata_5001-5075.jsonl!


In [17]:
# load jsonl files in latest_movies directory
import glob 
import json

movie_vi_data = []
for file in glob.glob(os.path.join(LATEST_MOVIES_DIR, "*.jsonl")):
    with open(file, "r", encoding="utf-8") as f:
        movie_vi_data.extend([json.loads(line) for line in f])

movie_vi_data_df = pd.DataFrame(movie_vi_data)
movie_vi_data_df.to_csv(os.path.join(LATEST_MOVIES_DIR, "plus_latest_movie_vi_data.csv"), index=False, encoding='utf-8-sig')


In [24]:
# lọc tmdbId trùng lặp và release_date < 2000
movie_vi_data_df = movie_vi_data_df.drop_duplicates(subset=['tmdbId'])
movie_vi_data_df = movie_vi_data_df[movie_vi_data_df['release_date'] >= '2000-01-01']

# phim overview_vi trống 
empty_overview_vi_df = movie_vi_data_df[movie_vi_data_df['overview_vi'].isnull() | (movie_vi_data_df['overview_vi'] == '')]
display(empty_overview_vi_df.head(100))
display(empty_overview_vi_df.count())

,tmdbId,title_vi,overview_vi,genres,release_date,poster_path,adult,backdrop_path,belongs_to_collection,budget,...,runtime,softcore,spoken_languages,status,tagline,title,video,vote_average,vote_count,movieId
7,1440098,Enfrentados: Marfil,,"[{'id': 10749, 'name': 'Phim Lãng Mạn'}, {'id'...",2026-09-09,/umlft1ztFbUcnHf960pXJocjpdv.jpg,False,/i65y7cMae36K0giN0GRaMjAHUru.jpg,"{'id': 1440105, 'name': 'Bilogía Enfrentados',...",0,...,104,False,"[{'english_name': 'Spanish', 'iso_639_1': 'es'...",Released,,Enfrentados: Marfil,False,6.356,52,None
9,1506560,Clash of the Thundermans,,"[{'id': 10770, 'name': 'Chương Trình Truyền Hì...",2026-09-03,/16oqRrWVzQm6qdGfBxvziZ2UiMT.jpg,False,/tBZMwg5oCJn8P0pStfuOMhhSQD3.jpg,"{'id': 1762706, 'name': 'The Thundermans Colle...",0,...,72,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,,Clash of the Thundermans,False,6.740,50,None
12,1516698,The Last Sunrise,,"[{'id': 10749, 'name': 'Phim Lãng Mạn'}, {'id'...",2026-08-26,/3PWJqDfygN0YNNjWsDUOXclCp3h.jpg,False,/zFCWfU2ViMIm38G1W8UsnSpqmWn.jpg,None,0,...,107,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,,The Last Sunrise,False,6.747,154,None
13,1621552,La captura,,"[{'id': 80, 'name': 'Phim Hình Sự'}, {'id': 28...",2026-08-21,/iwCeOpuBtuTP1kLosqgniey5OvX.jpg,False,/c4U96GrPRfnNrS01mBtqnocvlLJ.jpg,None,0,...,91,False,"[{'english_name': 'Portuguese', 'iso_639_1': '...",Released,,La captura,False,8.817,627,None
14,1323244,Rage of Stars,,"[{'id': 28, 'name': 'Phim Hành Động'}, {'id': ...",2026-08-06,/oLld47ZT1I3iecM3OWhIphohQUJ.jpg,False,/z7lZgL5tzefTfhyRtouThFhsuUS.jpg,None,0,...,106,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,,Rage of Stars,False,6.000,102,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
207,1036081,ഖലീഫ: The Ruler,,"[{'id': 80, 'name': 'Phim Hình Sự'}, {'id': 28...",2026-08-20,/cAl7uuRojdxysptYBj9Mg0czGmI.jpg,False,/6Jz7rnnZrSkgs9ySWUxewpcxr7Q.jpg,"{'id': 1754293, 'name': 'ഖലീഫ Collection', 'po...",0,...,157,False,"[{'english_name': 'Malayalam', 'iso_639_1': 'm...",Released,,ഖലീഫ: The Ruler,False,7.100,4,None
208,9539,Martyrs,,"[{'id': 27, 'name': 'Phim Kinh Dị'}, {'id': 18...",2008-06-12,/do92C3aiADF8SHC7gkRI65z6o9S.jpg,False,/fO1hwjlUqDe1ixO7eMfxTby62Ut.jpg,None,6500000,...,99,False,"[{'english_name': 'French', 'iso_639_1': 'fr',...",Released,,Martyrs,False,7.293,3121,None
209,1383233,The Penthouse,,"[{'id': 27, 'name': 'Phim Kinh Dị'}, {'id': 96...",2026-08-26,/AuBdbFZytyvFT6Tw2CRMRNxaoWt.jpg,False,/l9aNDMaOa7LFsF8WInhl6zkxT0E.jpg,None,0,...,102,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,,The Penthouse,False,0.000,0,None
212,1739202,Untold Raygun: Breaking Badly,,"[{'id': 99, 'name': 'Phim Tài Liệu'}]",2026-09-01,/3pnlJjsGtrUp3cPEOLzkR0sPQAK.jpg,False,/iBik58vWTbO3s9GgUW86v7fYqfZ.jpg,"{'id': 1745820, 'name': 'Untold Collection', '...",0,...,72,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,,Untold Raygun: Breaking Badly,False,6.042,12,None


tmdbId                   4735
title_vi                 4735
overview_vi              4735
genres                   4735
release_date             4735
poster_path              4034
adult                    4735
backdrop_path            2540
belongs_to_collection      83
budget                   4735
homepage                 4735
id                       4735
imdb_id                  2005
origin_country           4735
original_language        4735
original_title           4735
overview                 4735
popularity               4735
production_companies     4735
production_countries     4735
revenue                  4735
runtime                  4735
softcore                 4735
spoken_languages         4735
status                   4735
tagline                  4735
title                    4735
video                    4735
vote_average             4735
vote_count               4735
movieId                     0
dtype: int64

In [21]:
display(movie_vi_data_df.head(1000))

display(movie_vi_data_df.info())

display(movie_vi_data_df.count())

,tmdbId,title_vi,overview_vi,genres,release_date,poster_path,adult,backdrop_path,belongs_to_collection,budget,...,runtime,softcore,spoken_languages,status,tagline,title,video,vote_average,vote_count,movieId
0,969681,Người Nhện: Khởi Đầu Mới,"Không còn Tony Stark, MJ hay Ned kề cận, Peter...","[{'id': 878, 'name': 'Phim Khoa Học Viễn Tưởng...",2026-07-29,/wqGZVSCUSXE92WH2zyol2REaqT4.jpg,False,/qeQJx07rK2xm8SD2sJxFKhE7gs0.jpg,"{'id': 531241, 'name': 'Người Nhện', 'poster_p...",225000000,...,150,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,,Người Nhện: Khởi Đầu Mới,False,7.844,2641,None
1,1204680,Coyote vs. Acme,Sau quá nhiều lần thất bại vì những món hàng l...,"[{'id': 35, 'name': 'Phim Hài'}, {'id': 12, 'n...",2026-08-20,/vhv7lBWYM0DUuNU2a0V7Rhq21dD.jpg,False,/l9mFW9HQnAZ4r1ChZJHoOT3jaal.jpg,None,72000000,...,103,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,,Coyote vs. Acme,False,7.621,336,None
2,1368337,The Odyssey,Câu chuyện theo chân Odysseus trong hành trình...,"[{'id': 12, 'name': 'Phim Phiêu Lưu'}, {'id': ...",2026-07-15,/7876PK6gosE13Uq1B6nRxf7AJLT.jpg,False,/RMXG8myu1aGlNUsRjtxzmpdMK0.jpg,None,250000000,...,173,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Thách thức thần linh.,The Odyssey,False,8.005,3652,None
3,1288445,Tàu Buôn Người,Sau khi chứng kiến ​​vụ sát hại người chủ tỷ p...,"[{'id': 28, 'name': 'Phim Hành Động'}, {'id': ...",2026-08-19,/nDcgl5K2MCZk3xK3bh383CmIZsS.jpg,False,/e2QAGrEmbpmZpMymDRkDisJkvg9.jpg,None,40000000,...,95,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,,Tàu Buôn Người,False,6.426,373,None
4,1375646,Bầy Xác Sống,Giáo sư Se Jeong tham dự một hội nghị công ngh...,"[{'id': 28, 'name': 'Phim Hành Động'}, {'id': ...",2026-05-21,/lzI70txBtLH5kCUixYMJQffKkL9.jpg,False,/hpBGCnzOvdtQoMyE48gvwp2y5yx.jpg,None,0,...,123,False,"[{'english_name': 'Korean', 'iso_639_1': 'ko',...",Released,,Bầy Xác Sống,False,8.074,773,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1002,1739853,Мастер,,"[{'id': 99, 'name': 'Phim Tài Liệu'}]",2026-08-07,NaN,False,NaN,None,0,...,52,False,"[{'english_name': 'Russian', 'iso_639_1': 'ru'...",Released,,Мастер,False,0.000,0,None
1003,1744846,anywhere,,"[{'id': 18, 'name': 'Phim Chính Kịch'}]",2026-08-22,/7JDtIuzuhgGs5zhDa7OSw091kLl.jpg,False,/3FuBC9O2Q9DWSevMpnshflYByVa.jpg,None,6000,...,13,False,[],Released,,anywhere,False,0.000,0,None
1004,1748756,Gaskins: The Hitchhiker Killer,,"[{'id': 27, 'name': 'Phim Kinh Dị'}, {'id': 53...",2026-08-28,/31hhDTGKK9PFzFCorfL0mypVlh.jpg,False,/oD24mHAPRb4sPxg2K8CBVoIBqh3.jpg,None,0,...,94,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,,Gaskins: The Hitchhiker Killer,False,5.000,3,None
1005,1727438,Slick II,,"[{'id': 35, 'name': 'Phim Hài'}, {'id': 80, 'n...",2026-08-28,/fGUCH5UdXQcNnDHSfxIxNYhBCUN.jpg,False,NaN,None,69,...,30,False,[],Released,,Slick II,False,10.000,1,None


<class 'pandas.DataFrame'>
Index: 4927 entries, 0 to 5073
Data columns (total 31 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   tmdbId                 4927 non-null   int64  
 1   title_vi               4927 non-null   str    
 2   overview_vi            4927 non-null   str    
 3   genres                 4927 non-null   object 
 4   release_date           4927 non-null   str    
 5   poster_path            4226 non-null   str    
 6   adult                  4927 non-null   bool   
 7   backdrop_path          2729 non-null   str    
 8   belongs_to_collection  130 non-null    object 
 9   budget                 4927 non-null   int64  
 10  homepage               4927 non-null   str    
 11  id                     4927 non-null   int64  
 12  imdb_id                2188 non-null   str    
 13  origin_country         4927 non-null   object 
 14  original_language      4927 non-null   str    
 15  original_title      

None

tmdbId                   4927
title_vi                 4927
overview_vi              4927
genres                   4927
release_date             4927
poster_path              4226
adult                    4927
backdrop_path            2729
belongs_to_collection     130
budget                   4927
homepage                 4927
id                       4927
imdb_id                  2188
origin_country           4927
original_language        4927
original_title           4927
overview                 4927
popularity               4927
production_companies     4927
production_countries     4927
revenue                  4927
runtime                  4927
softcore                 4927
spoken_languages         4927
status                   4927
tagline                  4927
title                    4927
video                    4927
vote_average             4927
vote_count               4927
movieId                     0
dtype: int64